In [ ]:
import pandas as pd
from glob import glob
from datetime import datetime
from os.path import join as opj
import pickle
import itertools
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import os
import altair as alt

tqdm.pandas()

In [ ]:
def decompose_affine_matrix(matrix):
    """
    Decomposes a 2x3 or 3x3 affine transformation matrix into:
    rotation (theta in degrees), scale (sx, sy), shear (shear in degrees), translation (tx, ty).

    Args:
        matrix (numpy.ndarray): A 2x3 or 3x3 affine transformation matrix.

    Returns:
        dict: A dictionary containing 'rotation', 'scale_x', 'scale_y', 'shear', 'tx', 'ty'.
    """
    # Ensure matrix is 3x3
    if matrix.shape == (2, 3):
        matrix = np.vstack([matrix, [0, 0, 1]])
    elif matrix.shape != (3, 3):
        raise ValueError("Input matrix must be 2x3 or 3x3.")

    A = matrix[:2, :2]  # Extract the 2x2 transformation submatrix
    tx, ty = matrix[:2, 2]  # Extract translation components

    # Compute scale factors (remove shear effects)
    sx = np.linalg.norm(A[:, 0])  # Length of first column

    theta = np.arctan2(A[1, 0], A[0, 0])
    msy = A[0,1] * np.cos(theta) + A[1,1] * np.sin(theta)


    if np.sin(theta) == 0:
        sy = (A[1,1] - msy * np.sin(theta)) / np.cos(theta)
    else:
        sy = (msy * np.cos(theta) - A[0,1]) / np.sin(theta)

    shear = msy / sy

    return {
        "rotation": np.degrees(theta),
        "scale_x": sx,
        "scale_y": sy,
        "shear": np.degrees(shear),
        "tx": tx,
        "ty": ty
    }


In [ ]:
accepted_blocks = pd.read_csv(
    "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/histreg/out/accepted.csv")
alignment_results_dir = "/nfs/turbo/umms-tocho-snr/exp/chengjia/"

In [ ]:
def proc_row(x):
    with open(opj(
        alignment_results_dir, x["which"], f"{x['block']}_align.pkl"
    ), "rb") as fd:
        alignment_result = pickle.load(fd)

    decomposed = pd.concat([pd.DataFrame([decompose_affine_matrix(j) for j in i]) for i in alignment_result["matrices"]])
    decomposed["scale"] = decomposed[["scale_x", "scale_y"]].max(axis=1) / decomposed[["scale_x", "scale_y"]].min(axis=1)
    return decomposed["scale_x"] / decomposed["scale_y"], decomposed["shear"], decomposed["scale"].max(), decomposed["shear"].abs().max()

In [ ]:
accepted_blocks["scale_shear"]=accepted_blocks.progress_apply(proc_row, axis=1)

In [ ]:
accepted_blocks[["scales", "shears", "scale_max", "shear_max"]] = pd.DataFrame(accepted_blocks['scale_shear'].tolist(), index=accepted_blocks.index)
accepted_blocks = accepted_blocks.drop(["scale_shear","scale_max", "shear_max"], axis=1)

In [ ]:
all_points = accepted_blocks.explode(["scales", "shears"]).reset_index()

In [ ]:
to_viz = all_points[((all_points["scales"] - 1).abs() > 0.1) | ((all_points["shears"]).abs() > 2)]
to_viz

In [ ]:
#class_selection = alt.selection_point(fields=["exp_eval"], bind='legend')
#op = alt.condition( class_selection, alt.value(1.0), alt.value(0.1))
chart = alt.Chart(to_viz).mark_point().encode(
    x=alt.X('scales', axis=alt.Axis(tickSize=0)),
    y=alt.Y("shears", axis=alt.Axis(tickSize=0)),
    tooltip=["scales", "shears","block", "which", "status"],
    #opacity=op,
).properties(
    width=700,
    height=700,
    title='scale vs shear on affine xform'
).configure_axis(
    labelFontSize=16,
    titleFontSize=16
).interactive()#.add_params(
#    class_selection
#).interactive()


In [ ]:
chart

In [ ]:
with open(opj(
        alignment_results_dir, x["which"], f"{x['block']}_align.pkl"
    ), "rb") as fd:
        alignment_result = pickle.load(fd)

decomposed = pd.concat([pd.DataFrame([decompose_affine_matrix(j) for j in i]) for i in alignment_result["matrices"]])
decomposed["scale"] = decomposed[["scale_x", "scale_y"]].max(axis=1) / decomposed[["scale_x", "scale_y"]].min(axis=1)
decomposed

In [ ]:
patched = pd.read_csv(f"/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/SU-23-67707.D1_blockmeta.csv")

In [ ]:
patched

In [ ]:
import tifffile
import cv2
import torch
from itertools import chain, accumulate, compress

def sanitize_string(string):
    return re.sub(r'[^a-zA-Z0-9]', '', string)


def get_sections_annot(he_annot):
    annot_mask = np.all(he_annot == np.array([[[0, 255, 0]]]), axis=2)
    contours, _ = cv2.findContours(annot_mask.astype(np.uint8),
                                   cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return [None]

    # Create a filled mask for each contour
    masked_ims = []
    for i, contour in enumerate(contours):
        # Create an empty mask
        mask = np.zeros_like(annot_mask, dtype=float)
        # Fill the contour
        cv2.drawContours(mask, [contour], -1, 1, thickness=cv2.FILLED)
        masked_ims.append(torch.tensor(mask))

    return masked_ims

def recreate_block_meta_from_annot(su_b):
    mask_annot_meta = pd.read_csv(opj(sections_annot_root, f"meta/{su_b}_sections_annot_meta.csv"))

    mask_annot = tifffile.imread(
           opj(sections_annot_root, f"block_annot/{su_b}.tiff"))

    #if len(mask_annot.shape) == 3: # THIS IS THE BUGGY IMOLEMENTATION THAT IS IN BLOCK PATCH
    #    mask_annot = np.expand_dims(mask_annot, 0)

    section_mask = [get_sections_annot(m) for m in mask_annot]


    to_remove = (mask_annot_meta["comment"] == "RM")
    mask_annot_meta=mask_annot_meta[~to_remove].reset_index(              drop=True)
    section_mask = list(compress(section_mask, ~to_remove))

    flat_mask = list(chain.from_iterable(section_mask))
    lengths = list(map(len, section_mask))  # Lengths of each sublist
    starts = [0] + list(accumulate(
        lengths[:-1]))  # Start indices of each sublist in the flattened list
    list_indices = [
        list(range(start, start + length))
        for start, length in zip(starts, lengths)
    ]

    mask_annot_meta["sections_mask"] = list_indices
    curr_block_recreated = mask_annot_meta.explode("sections_mask").reset_index(
            drop=False).rename({"index": "svs_idx"}, axis=1)
    return curr_block_recreated

In [ ]:
for _, b in tqdm(accepted_blocks.iterrows()):
    if not os.path.exists(f"/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/{b['block']}_blockmeta.csv"):
        continue

    patched = pd.read_csv(f"/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/{b['block']}_blockmeta.csv")
    annot = pd.read_csv(f"/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/pixel_alignment/sections_annot2/meta/{b['block']}_sections_annot_meta.csv")

    if not (patched[["Stain", "path"]].drop_duplicates().reset_index(drop=True).equals(annot[["Stain", "path"]].drop_duplicates().reset_index(drop=True))):

        print(patched)
        print("---")
        print(annot)
        print("===")

In [ ]:
to_re_patch = []

for _, b in tqdm(accepted_blocks.iterrows()):
    with open(opj(
            alignment_results_dir, b["which"], f"{b['block']}_align.pkl"
        ), "rb") as fd:
            alignment_result = pickle.load(fd)

    align_shape = alignment_result["matrices"].shape


    rbm = recreate_block_meta_from_annot(b['block'])
    num_he = (rbm["Stain"]=="H&E").sum()
    num_tgt = len(rbm["Stain"])

    if not ((align_shape[0] == num_he) and (align_shape[1] == num_tgt)):
        #print(rbm)
        #print("---")
        #print(align_shape)
        #print("===")
        to_re_patch.append(b)
    #import pdb; pdb.set_trace()

In [ ]:
sections_annot_root = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/pixel_alignment/sections_annot2/"
su_b = "SU-23-20287.BFS1"

In [ ]:
mask_annot_meta = pd.read_csv(opj(sections_annot_root, f"meta/{su_b}_sections_annot_meta.csv"))



In [ ]:
pd.DataFrame(to_re_patch)

In [ ]:
to_re_patch2 = to_re_patch

In [ ]:
accepted = pd.read_csv("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/histreg/out/accepted.csv")
one_sec = pd.read_csv("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/histreg/out/one_section.csv")


In [ ]:
patched = glob("/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/*.csv")
patched = [p.removeprefix("/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/").removesuffix("_blockmeta.csv") for p in patched]

In [ ]:
set(patched).difference(set(accepted["block"].tolist())).difference(set(one_sec["block"].tolist()))